# Tutorial 100 — Persist retrieval indexes

The previous tutorials built semantic, lexical, and code-aware retrieval
structures in memory. This tutorial stores those structures together and loads
them again without recomputing embeddings or repository relationships.

A fingerprint records the inputs that determine whether the saved retrieval
data is still valid. It is therefore a **cache-validity mechanism**, not merely
metadata: when the relevant chunks or graph change, QAChat can detect that the
saved index is stale and rebuild it.

Tutorial 110 will use the three retrieval channels together and combine their
rankings with Reciprocal Rank Fusion (RRF).


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 2. Load an embedding model

A semantic index must use the same embedding model whenever its vectors are
built or queried. As in Tutorials 60 and 70, QAChat resolves the backend first
and selects the matching canonical embedding-model default.


In [ ]:
from qachat import (
    BACKEND_AUTO,
    DEFAULT_EMBEDDING_MODEL_MLX,
    DEFAULT_EMBEDDING_MODEL_TORCH,
    choose_backend,
)


BACKEND = BACKEND_AUTO
selected_backend = choose_backend(BACKEND)
MODEL_ID = (
    DEFAULT_EMBEDDING_MODEL_MLX
    if selected_backend == "mlx"
    else DEFAULT_EMBEDDING_MODEL_TORCH
)
ALLOW_DOWNLOAD = True

print(f"Requested backend: {BACKEND}")
print(f"Resolved backend: {selected_backend}")
print(f"Embedding model: {MODEL_ID}")


In [ ]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 3. Prepare repository chunks

Prepare the same source files and token-bounded chunks that will feed all three
retrieval structures. Persistence is only useful when the saved structures can
be tied back to the exact repository representation that produced them.


In [ ]:
from qachat import scan_repository


report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)


In [ ]:
from qachat import extract_source_units, read_repository_files


source_files = read_repository_files(report)
units = extract_source_units(source_files)


In [ ]:
from qachat import chunk_source_units


chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 4. Build the retrieval structures

QAChat creates three reusable views of the same repository:

- a semantic index for meaning;
- a lexical index for wording and identifiers;
- a code graph for deterministic source relationships.

Keeping them together allows later hybrid retrieval to reuse a consistent
snapshot of the repository.


In [ ]:
from qachat import build_semantic_index


semantic_index = build_semantic_index(chunks, embedding_session)


In [ ]:
from qachat import build_lexical_index


lexical_index = build_lexical_index(chunks)


In [ ]:
from qachat import build_code_graph


code_graph = build_code_graph(source_files, units)


## 5. Save all retrieval data

`prepare_persistent_indexes()` packages the three retrieval structures with a
fingerprint. `save_indexes()` then writes that package as ordinary JSON.

The fingerprint is the cache-validity contract: it summarizes the repository
representation and index-relevant settings so QAChat can later determine whether
a saved index still corresponds to the current inputs.


In [ ]:
from qachat import prepare_persistent_indexes


indexes = prepare_persistent_indexes(
    semantic_index,
    lexical_index,
    code_graph,
)

print(f"Fingerprint: {indexes.fingerprint}")


In [ ]:
from qachat import save_indexes


INDEX_PATH = root / "sandbox" / "indexes" / "qachat.json"

saved_path = save_indexes(indexes, INDEX_PATH)

print(f"Saved: {saved_path}")
print(f"Size: {saved_path.stat().st_size:,} bytes")


The saved JSON contains the chunks, semantic vectors, lexical vectors, code
graph, format version, and fingerprint needed to reconstruct the retrieval
state. The following preview deliberately shows only a small sample from each
part so the persistent format is inspectable without printing the full index.


In [ ]:
import json


payload = json.loads(INDEX_PATH.read_text(encoding="utf-8"))

preview = {
    "format_version": payload["format_version"],
    "fingerprint": payload["fingerprint"][:16] + "...",
    "chunk": {
        "path": payload["chunks"][0]["path"],
        "kind": payload["chunks"][0]["kind"],
        "symbol": payload["chunks"][0]["symbol"],
    },
    "semantic_embedding": payload["semantic_embeddings"][0][:5],
    "lexical_vector": dict(
        list(payload["lexical_vectors"][0].items())[:5]
    ),
    "code_entity": payload["code_graph"]["entities"][0],
    "code_relation": payload["code_graph"]["relations"][0],
}

print(json.dumps(preview, indent=2))

## 6. Load and validate the saved index

Loading restores all three structures without rebuilding embeddings, TF-IDF
weights, or code relationships. `indexes_are_current()` then recomputes the
validity conditions against the current chunks and graph.

The final deliberate text change demonstrates the purpose of the fingerprint:
a changed repository representation makes the previously saved retrieval data
stale, so it should not be reused silently.


In [ ]:
from qachat import load_indexes


loaded_indexes = load_indexes(INDEX_PATH)

print(f"Loaded semantic chunks: {len(loaded_indexes.semantic.chunks)}")
print(f"Loaded lexical chunks: {len(loaded_indexes.lexical.chunks)}")
print(f"Loaded code entities: {len(loaded_indexes.code_graph.entities)}")
print(f"Loaded code relationships: {len(loaded_indexes.code_graph.relations)}")


In [ ]:
from qachat import indexes_are_current


assert loaded_indexes.code_graph == code_graph
assert indexes_are_current(loaded_indexes, chunks, code_graph)

print("Saved retrieval data matches the current repository.")


In [ ]:
from dataclasses import replace


changed_chunks = list(chunks)
changed_chunks[0] = replace(
    changed_chunks[0],
    text=changed_chunks[0].text + "\n# changed",
)

assert not indexes_are_current(loaded_indexes, changed_chunks, code_graph)

print("A repository change makes the saved data stale.")


## 7. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Continue the tutorial series

**Previous:** [Tutorial 90 — Retrieve through code relationships](./90_code_aware_retrieval.ipynb)

**Next:** [Tutorial 110 — Combine retrieval channels](./110_hybrid_ranking.ipynb)
